In [ ]:
!pip install -q "transformers>=4.46.0" accelerate sentencepiece pandas qwen-vl-utils gdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 24.4 MB/s eta 0:00:00


In [ ]:
import os, json, time, re
from pathlib import Path
from tqdm import tqdm
from PIL import Image
import torch
import torch.nn.functional as F
import numpy as np
from transformers import LogitsProcessor, LogitsProcessorList

ON_KAGGLE = os.path.exists("/kaggle")
BASE_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("/content")
AMBER_DIR = BASE_DIR / "AMBER"
IMG_DIR = AMBER_DIR / "image"

if not (AMBER_DIR / "data").exists():
    !git clone https://github.com/junyangwang0410/AMBER.git {AMBER_DIR}

if not IMG_DIR.exists() or len(list(IMG_DIR.glob("AMBER_*.jpg"))) < 1000:
    import gdown
    url = "https://drive.google.com/uc?id=1MaCHgtupcZUjf007anNl4_MV0o4DjXvl"
    gdown.download(url, str(BASE_DIR / "AMBER_images.zip"), quiet=False)
    !unzip -q {BASE_DIR / "AMBER_images.zip"} -d {AMBER_DIR}
    for candidate in [AMBER_DIR, AMBER_DIR / "AMBER", AMBER_DIR / "image"]:
        if candidate.is_dir() and len(list(candidate.glob("AMBER_*.jpg"))) > 100:
            IMG_DIR = candidate
            break

query_rel = json.load(open(AMBER_DIR / "data/query/query_discriminative-relation.json"))
annotations = json.load(open(AMBER_DIR / "data/annotations.json"))

gt_map, subtype_map = {}, {}
for ann in annotations:
    truth = ann.get("truth")
    if isinstance(truth, str):
        gt_map[ann["id"]] = truth.strip().lower()
        subtype_map[ann["id"]] = ann.get("type")

Cloning into '/kaggle/working/AMBER'...
remote: Enumerating objects: 41, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 41 (delta 16), reused 21 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (41/41), 874.48 KiB | 6.57 MiB/s, done.
Resolving deltas: 100% (16/16), done.


Downloading...
From (original): https://drive.google.com/uc?id=1MaCHgtupcZUjf007anNl4_MV0o4DjXvl
From (redirected): https://drive.google.com/uc?id=1MaCHgtupcZUjf007anNl4_MV0o4DjXvl&confirm=t&uuid=f1a6fc08-87e5-4137-8cd1-eba97e02aa71
To: /kaggle/working/AMBER_images.zip
100%|██████████| 415M/415M [00:04<00:00, 99.9MB/s]


In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

MODEL_TAG = "qwen2_2b"
MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"

# Critical fix: capping max_pixels prevents the >12s/iteration slowdown on large AMBER images
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 512 * 28 * 28

processor = AutoProcessor.from_pretrained(MODEL_NAME, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
)
model.eval()

preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

Qwen2VLForConditionalGeneration(
  (model): Qwen2VLModel(
    (visual): Qwen2VisionTransformerPretrainedModel(
      (patch_embed): PatchEmbed(
        (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
      )
      (rotary_pos_emb): VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-31): 32 x Qwen2VLVisionBlock(
          (norm1): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
          (attn): VisionAttention(
            (qkv): Linear(in_features=1280, out_features=3840, bias=True)
            (proj): Linear(in_features=1280, out_features=1280, bias=True)
          )
          (mlp): VisionMlp(
            (fc1): Linear(in_features=1280, out_features=5120, bias=True)
            (act): QuickGELUActivation()
            (fc2): Linear(in_features=5120, out_features=1280, bias=True)
          )
        )
      )
      (merger): PatchMerger(
        (ln_q): LayerN

In [ ]:
from qwen_vl_utils import process_vision_info

NEG_RE = re.compile(r"\b(not|no|none|isn't|aren't|doesn't|don't|wasn't|weren't|isnt|arent|doesnt|dont|wasnt|werent|n't)\b")

def parse_yes_no(response):
    r = response.strip().lower()
    if r.startswith("yes"): return "yes"
    if r.startswith("no"): return "no"
    for word in re.findall(r"[a-z']+", r):
        if word in ("yes", "no"): return word
    if NEG_RE.search(r): return "no"
    return None

class ContrastiveAPCLogitsProcessor(LogitsProcessor):
    def __init__(self, model, cf_prompt_ids, prompt_len_standard, alpha=1.0, beta=0.1):
        self.model = model
        self.cf_prompt_ids = cf_prompt_ids
        self.prompt_len_standard = prompt_len_standard
        self.alpha = alpha
        self.beta = beta

    def __call__(self, input_ids, scores):
        generated_so_far = input_ids[:, self.prompt_len_standard:]
        cf_input_ids = torch.cat([self.cf_prompt_ids, generated_so_far], dim=1)

        with torch.no_grad():
            cf_out = self.model(input_ids=cf_input_ids, use_cache=False)
        cf_logits = cf_out.logits[:, -1, :].float()

        standard_logits = scores.float()
        standard_probs = F.softmax(standard_logits, dim=-1)
        threshold = self.beta * standard_probs.max(dim=-1, keepdim=True).values
        apc_mask = standard_probs >= threshold

        cd_logits = (1 + self.alpha) * standard_logits - self.alpha * cf_logits
        cd_logits = cd_logits.masked_fill(~apc_mask, float("-inf"))
        return cd_logits

def build_standard_inputs(image, question):
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": question + " Answer with only 'Yes' or 'No'."}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    return processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(model.device)

def build_cf_prompt_ids(question):
    messages = [{"role": "user", "content": [{"type": "text", "text": question + " Answer with only 'Yes' or 'No'."}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return processor.tokenizer(text, return_tensors="pt").input_ids.to(model.device)

@torch.no_grad()
def get_baseline_with_entropy(image, question, max_new_tokens=6):
    inputs = build_standard_inputs(image, question)
    outputs = model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False,
        return_dict_in_generate=True, output_scores=True
    )
    first_token_logits = outputs.scores[0][0]
    probs = F.softmax(first_token_logits.float(), dim=-1)
    entropy = -torch.sum(probs * torch.log(probs + 1e-9)).item()

    gen_ids = outputs.sequences[0][inputs["input_ids"].shape[1]:]
    response = processor.decode(gen_ids, skip_special_tokens=True).strip()
    return response, entropy

@torch.no_grad()
def generate_contrastive(image, question, alpha=1.0, beta=0.1, max_new_tokens=6):
    standard_inputs = build_standard_inputs(image, question)
    cf_prompt_ids = build_cf_prompt_ids(question)
    prompt_len_standard = standard_inputs["input_ids"].shape[1]

    processor_list = LogitsProcessorList([
        ContrastiveAPCLogitsProcessor(model, cf_prompt_ids, prompt_len_standard, alpha=alpha, beta=beta)
    ])
    out_ids = model.generate(
        **standard_inputs, max_new_tokens=max_new_tokens, do_sample=False,
        logits_processor=processor_list,
    )
    gen = out_ids[0][prompt_len_standard:]
    return processor.decode(gen, skip_special_tokens=True).strip()

def generate_all_signals_entropy(image, question, alpha=1.0, beta=0.1):
    baseline_resp, entropy = get_baseline_with_entropy(image, question)
    cd_resp = generate_contrastive(image, question, alpha, beta)
    return parse_yes_no(baseline_resp), parse_yes_no(cd_resp), entropy

In [ ]:
# --- Sanity check: does the model actually emit clean yes/no? ---
N_CHECK = 5  # bump this up if you want a bigger sample

sanity_rows = []
counts = {"yes": 0, "no": 0, "other": 0}

for i in range(min(N_CHECK, len(query_rel))):
    q = query_rel[i]
    img_path = IMG_DIR / q["image"]
    if not img_path.exists():
        continue

    image = Image.open(img_path).convert("RGB")
    image.thumbnail((512, 512))

    inputs = build_standard_inputs(image, q["query"])

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
        )

    # strip the prompt tokens, decode only the newly generated part
    gen_ids = out_ids[:, inputs["input_ids"].shape[1]:]
    raw_text = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]

    parsed = parse_yes_no(raw_text)
    bucket = parsed if parsed in ("yes", "no") else "other"
    counts[bucket] += 1

    sanity_rows.append({
        "id": q["id"],
        "query": q["query"],
        "raw_output": raw_text,
        "parsed": parsed,
    })

    print(f"[{i}] raw={raw_text!r:40} parsed={parsed}")

print("\n--- Summary ---")
print(counts)
if counts["other"] > 0:
    print(f"⚠️ {counts['other']}/{len(sanity_rows)} outputs did NOT parse to yes/no — inspect the raw_output values above.")
else:
    print("✅ All sampled outputs parsed cleanly to yes/no.")

[0] raw='Yes'                                    parsed=yes
[1] raw='No'                                     parsed=no
[2] raw='No'                                     parsed=no
[3] raw='Yes'                                    parsed=yes
[4] raw='Yes'                                    parsed=yes

--- Summary ---
{'yes': 3, 'no': 2, 'other': 0}
✅ All sampled outputs parsed cleanly to yes/no.


In [ ]:
ALPHA = 1.0
BETA = 0.1

CHECKPOINT_PATH = str(BASE_DIR / f"sira_entropy_gated_{MODEL_TAG}.json")
results = []
start_idx = 0

if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        results = json.load(f)
    start_idx = len(results)
    print(f"Resuming from sample {start_idx}")

for i in tqdm(range(start_idx, len(query_rel)), desc=f"Entropy Gating ({MODEL_TAG})"):
    q = query_rel[i]
    img_path = IMG_DIR / q["image"]
    gt = gt_map.get(q["id"])
    subtype = subtype_map.get(q["id"])

    if not img_path.exists() or gt is None: continue

    # Pre-resize image to protect T4 VRAM from token explosion
    image = Image.open(img_path).convert("RGB")
    image.thumbnail((512, 512))

    try:
        base_pred, cd_pred, entropy_score = generate_all_signals_entropy(image, q["query"], alpha=ALPHA, beta=BETA)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        base_pred, cd_pred, entropy_score = None, None, 0.0

    results.append({
        "id": q["id"], "subtype": subtype, "gt": gt,
        "baseline_pred": base_pred, "cd_pred": cd_pred, "entropy": entropy_score
    })

    if (i + 1) % 50 == 0:
        with open(CHECKPOINT_PATH, "w") as f:
            json.dump(results, f)

with open(CHECKPOINT_PATH, "w") as f:
    json.dump(results, f)

def calc_metrics_entropy(res, threshold):
    tp = fp = tn = fn = 0
    for r in res:
        pred = r["cd_pred"] if r["entropy"] >= threshold else r["baseline_pred"]
        gt = r["gt"]

        if r["subtype"] == "discriminative-relation" and gt == "yes":
            if pred == "yes": tp += 1
            else: fn += 1
        elif r["subtype"] == "relation" and gt == "no":
            if pred == "no": tn += 1
            else: fp += 1

    recall = tp / (tp + fn) * 100 if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) * 100 if (tn + fp) > 0 else 0
    bal_acc = (recall + spec) / 2
    return recall, spec, bal_acc

print(f"\n=== ENTROPY-GATED SYSTEM 2 DECODING: PARETO FRONTIER ({MODEL_TAG}) ===")
print(f"{'Threshold':<10} | {'Recall':<10} | {'Specificity':<15} | {'Balanced Acc':<15}")
print("-" * 55)

rec_base, spec_base, bal_base = calc_metrics_entropy(results, threshold=999.0)
rec_cd, spec_cd, bal_cd = calc_metrics_entropy(results, threshold=-1.0)
print(f"{'Baseline':<10} | {rec_base:<10.2f} | {spec_base:<15.2f} | {bal_base:<15.2f}")
print(f"{'Pure CD+APC':<10} | {rec_cd:<10.2f} | {spec_cd:<15.2f} | {bal_cd:<15.2f}")
print("-" * 55)

sweep_data = []
for t in np.linspace(0.55, 0.75, 21):
    rec, spec, bal_acc = calc_metrics_entropy(results, t)
    sweep_data.append({"threshold": t, "recall": rec, "specificity": spec, "balanced_acc": bal_acc})
    print(f"{t:<10.3f} | {rec:<10.2f} | {spec:<15.2f} | {bal_acc:<15.2f}")

# --- Final Save & Download Block ---
final_output = {
    "model": MODEL_TAG,
    "alpha": ALPHA,
    "beta": BETA,
    "baseline": {"recall": rec_base, "specificity": spec_base, "balanced_acc": bal_base},
    "cd_apc": {"recall": rec_cd, "specificity": spec_cd, "balanced_acc": bal_cd},
    "entropy_threshold_sweep": sweep_data,
    "raw_results": results
}

FINAL_PATH = str(BASE_DIR / f"final_entropy_gated_results_{MODEL_TAG}.json")
with open(FINAL_PATH, "w") as f:
    json.dump(final_output, f, indent=2)

print(f"\nSaved complete results and sweep data to {FINAL_PATH}")

if not ON_KAGGLE:
    from google.colab import files
    files.download(FINAL_PATH)

Entropy Gating (qwen2_2b): 100%|██████████| 1664/1664 [18:41<00:00,  1.48it/s]


=== ENTROPY-GATED SYSTEM 2 DECODING: PARETO FRONTIER (qwen2_2b) ===
Threshold  | Recall     | Specificity     | Balanced Acc   
-------------------------------------------------------
Baseline   | 48.82      | 94.05           | 71.43          
Pure CD+APC | 75.79      | 77.65           | 76.72          
-------------------------------------------------------
0.550      | 75.79      | 78.23           | 77.01          
0.560      | 75.79      | 78.23           | 77.01          
0.570      | 75.79      | 78.23           | 77.01          
0.580      | 75.79      | 78.52           | 77.16          
0.590      | 75.79      | 78.52           | 77.16          
0.600      | 75.79      | 78.66           | 77.23          
0.610      | 75.79      | 78.66           | 77.23          
0.620      | 75.79      | 79.10           | 77.45          
0.630      | 75.69      | 79.54           | 77.61          
0.640      | 75.69      | 79.54           | 77.61          
0.650      | 75.69      | 81.13       

In [ ]:
from google.colab import files
files.download(FINAL_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>